In [ ]:
# -*- coding: utf-8 -*-
"""Hotel Review Sentiment Analysis with BERT (FULL DATASET).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1_your_drive_link_here
"""

# Install required packages
!pip install transformers torch langdetect num2words
!pip install accelerate -U
!pip install openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect, DetectorFactory
import re
from num2words import num2words
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
DetectorFactory.seed = 0
torch.manual_seed(42)
np.random.seed(42)

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Load the data
file_path = '/content/drive/MyDrive/Colab Notebooks/hotel.xlsx'
df = pd.read_excel(file_path)
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")

# Remove the 'Unnamed: 10' column if it exists and is empty
if 'Unnamed: 10' in df.columns and df['Unnamed: 10'].isnull().all():
    df = df.drop(columns=['Unnamed: 10'])
    print("Removed 'Unnamed: 10' column as it was empty.")
    print(f"New Shape: {df.shape}")


# Data Cleaning Function (same as VADER)
def clean_review_data(df):
    clean_df = df.copy()

    # Handle missing values
    initial_count = len(clean_df)
    clean_df = clean_df.dropna(subset=['Review Summary'])
    print(f"Removed {initial_count - len(clean_df)} rows with missing reviews")

    # Remove duplicates
    initial_count = len(clean_df)
    clean_df = clean_df.drop_duplicates(subset=['Review Summary'])
    print(f"Removed {initial_count - len(clean_df)} duplicate reviews")

    # Filter English reviews
    def is_english(text):
        try:
            return detect(str(text)) == 'en'
        except:
            return False

    initial_count = len(clean_df)
    clean_df = clean_df[clean_df['Review Summary'].apply(is_english)]
    print(f"Removed {initial_count - len(clean_df)} non-English reviews")

    # Text cleaning
    clean_df['Review Summary'] = clean_df['Review Summary'].str.lower()

    abbreviation_map = {
        r'\bbr\b': 'bathroom', r'\bw\/': 'with', r'\bw\/o\b': 'without',
        r'\brm\b': 'room', r'\bbf\b': 'breakfast', r'\bac\b': 'air conditioning',
        r'\bwifi\b': 'wi-fi', r'\btv\b': 'television', r'\bmin\b': 'minute',
    }

    for abbrev, full_form in abbreviation_map.items():
        clean_df['Review Summary'] = clean_df['Review Summary'].str.replace(abbrev, full_form, regex=True)

    # Convert numbers to words
    def convert_numbers(text):
        def num_to_word(match):
            num = match.group()
            try:
                return num2words(int(num))
            except:
                return num
        return re.sub(r'\b\d+\b', num_to_word, text)

    clean_df['Review Summary'] = clean_df['Review Summary'].apply(convert_numbers)
    clean_df['Review Summary'] = clean_df['Review Summary'].str.replace(r'[^\w\s]', ' ', regex=True)
    clean_df['Review Summary'] = clean_df['Review Summary'].str.replace(r'\s+', ' ', regex=True)
    clean_df['Review Summary'] = clean_df['Review Summary'].str.strip()

    print(f"Final dataset size: {len(clean_df)} reviews")
    return clean_df

# Apply cleaning (SAME as VADER)
cleaned_df = clean_review_data(df)

# Load BERT Model
print("\n" + "=" * 60)
print("LOADING BERT SENTIMENT ANALYSIS MODEL")
print("=" * 60)

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
print(f"Loading model: {model_name}...")

# Initialize with smaller batch size for full dataset
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    device=device,
    truncation=True,
    max_length=256,  # Reduced for memory
    batch_size=8     # Smaller batches for full dataset
)

print("Model loaded successfully!")

# Function for BERT sentiment analysis with error handling
def bert_sentiment_analysis(texts, batch_size=8):
    results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        try:
            batch_results = sentiment_analyzer(batch_texts)
            results.extend(batch_results)
            if i % 100 == 0:  # Progress tracking
                print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} reviews")
        except Exception as e:
            print(f"Error processing batch {i//batch_size + 1}: {e}")
            # Add neutral sentiment for failed analyses
            results.extend([{'label': '3 stars', 'score': 0.5}] * len(batch_texts))

    return results

# Use ALL reviews (no sampling)
analysis_df = cleaned_df.copy()
print(f"Analyzing ALL {len(analysis_df)} reviews...")

# Perform sentiment analysis on ALL reviews
reviews_list = analysis_df['Review Summary'].astype(str).tolist()
sentiment_results = bert_sentiment_analysis(reviews_list)

# Extract star ratings
def extract_stars(label):
    return int(label.split()[0])

analysis_df['stars'] = [extract_stars(result['label']) for result in sentiment_results]
analysis_df['sentiment_score'] = [result['score'] for result in sentiment_results]
analysis_df['sentiment'] = analysis_df['stars'].apply(lambda x: 1 if x >= 4 else 0)

print("Sentiment analysis completed for ALL reviews!")

# Compare with VADER results (if available)
try:
    vader_path = '/content/drive/MyDrive/Colab Notebooks/hotel_reviews_with_sentiment.xlsx'
    vader_df = pd.read_excel(vader_path)

    print("\n" + "=" * 60)
    print("COMPARISON: VADER vs BERT")
    print("=" * 60)

    print(f"VADER total reviews: {len(vader_df)}")
    print(f"BERT total reviews: {len(analysis_df)}")

    # Hotel count comparison
    vader_hotels = vader_df['Hotel name'].value_counts()
    bert_hotels = analysis_df['Hotel name'].value_counts()

    print(f"\nVADER unique hotels: {len(vader_hotels)}")
    print(f"BERT unique hotels: {len(bert_hotels)}")

    # Check if same hotels
    common_hotels = set(vader_hotels.index) & set(bert_hotels.index)
    print(f"Common hotels: {len(common_hotels)}")

except FileNotFoundError:
    print("VADER results file not found for comparison")

# Save FULL results
output_path = '/content/drive/MyDrive/Colab Notebooks/hotel_reviews_bert_full_sentiment.xlsx'
analysis_df.to_excel(output_path, index=False)
print(f"\nFull results saved to: {output_path}")

# Analysis results
print("\n" + "=" * 60)
print("FINAL RESULTS - ALL REVIEWS")
print("=" * 60)

star_counts = analysis_df['stars'].value_counts().sort_index()
sentiment_counts = analysis_df['sentiment'].value_counts()

print("\nStar Rating Distribution:")
for stars, count in star_counts.items():
    print(f"{stars} stars: {count} reviews ({count/len(analysis_df)*100:.1f}%)")

print(f"\nSentiment Distribution:")
print(f"Positive reviews (4-5 stars): {sentiment_counts.get(1, 0)}")
print(f"Negative reviews (1-3 stars): {sentiment_counts.get(0, 0)}")

print("\nAnalysis complete! ✅ Now both VADER and BERT should have same review counts per hotel.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!
Shape: (2838, 11)
Removed 262 rows with missing reviews
Removed 701 duplicate reviews
Removed 158 non-English reviews
Final dataset size: 1717 reviews

LOADING BERT SENTIMENT ANALYSIS MODEL
Using device: CPU
Loading model: nlptown/bert-base-multilingual-uncased-sentiment...


Device set to use cpu


Model loaded successfully!
Analyzing ALL 1717 reviews...
Processed 8/1717 reviews
Processed 208/1717 reviews
Processed 408/1717 reviews
Processed 608/1717 reviews
Processed 808/1717 reviews
Processed 1008/1717 reviews
Processed 1208/1717 reviews
Processed 1408/1717 reviews
Processed 1608/1717 reviews
Sentiment analysis completed for ALL reviews!

COMPARISON: VADER vs BERT
VADER total reviews: 1717
BERT total reviews: 1717

VADER unique hotels: 13
BERT unique hotels: 13
Common hotels: 13

Full results saved to: /content/drive/MyDrive/Colab Notebooks/hotel_reviews_bert_full_sentiment.xlsx

FINAL RESULTS - ALL REVIEWS

Star Rating Distribution:
1 stars: 73 reviews (4.3%)
2 stars: 78 reviews (4.5%)
3 stars: 162 reviews (9.4%)
4 stars: 400 reviews (23.3%)
5 stars: 1004 reviews (58.5%)

Sentiment Distribution:
Positive reviews (4-5 stars): 1404
Negative reviews (1-3 stars): 313

Analysis complete! ✅ Now both VADER and BERT should have same review counts per hotel.
